# **Verification and Sensitivity Analyses**

Supplementary analyses for the study on computationally efficient pneumonia detection. The main notebooks are kept as the record of the experiments as they were run. This notebook adds checks that were run after the main experiments, without retraining any model.

| Section | Content |
|---|---|
| 1 | NIH patient overlap audit |
| 2 | 95% confidence intervals for the deployment-selected models |
| 3 | Multi-label split reconciliation |
| 4 | FP32 TFLite baselines: size and timing |
| 5 | Xception, original against INT8: paired comparison |
| 6 | Grad-CAM speed benchmark |
| 7 | Ensemble: comparison of two input pipelines |
| 8 | NIH test folder identity check |
| 9 | Grad-CAM at a 128 x 128 input |
| 10 | Training-protocol audit of the training notebooks |

Run the cells in order on a fresh runtime. The thread comparison for the TensorFlow Lite interpreter, the XNNPACK comparison and the parameter and MAC counts are in the Supplementary notebook. The ensemble weight analysis is in the Optimisation notebook.

## **Setup**

In [ ]:
# Thread limits must be set before TensorFlow is imported
import os
os.environ['TF_NUM_INTRAOP_THREADS'] = '4'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'

from google.colab import drive
drive.mount('/content/drive')

import glob, re, time
import numpy as np, pandas as pd, psutil, tensorflow as tf

DRIVE = '/content/drive/MyDrive'

# Clock adjustment used throughout the study: measured time x (host clock / 1.5 GHz)
cpu_ghz = psutil.cpu_freq().current / 1000
clock_factor = cpu_ghz / 1.5
print(f'Host clock {cpu_ghz:.2f} GHz, {psutil.cpu_count(logical=False)} physical and {psutil.cpu_count()} logical cores, '
      f'{psutil.virtual_memory().total / 1024**3:.2f} GB RAM')
print(f'TensorFlow threads: intra-op {tf.config.threading.get_intra_op_parallelism_threads()}, '
      f'inter-op {tf.config.threading.get_inter_op_parallelism_threads()}')
print(f'Clock adjustment factor: {clock_factor:.2f}')


def find_file(name):
    # First match for a file name anywhere under My Drive
    hits = glob.glob(f'{DRIVE}/**/{name}', recursive=True)
    assert hits, f'{name} not found under {DRIVE}'
    return hits[0]

Mounted at /content/drive
Host clock 2.20 GHz, 1 physical and 2 logical cores, 12.67 GB RAM
TensorFlow threads: intra-op 0, inter-op 0
Clock adjustment factor: 1.47


## **1. NIH patient overlap audit**

The NIH binary experiment split the images at image level. Each image is mapped to its patient through `Data_Entry_2017_v2020.csv`, and the patients present in both the training and test folders are counted. Augmented training files (prefix `aug_<n>_`) are traced back to their source image to see how many derive from a test image.

The image folders were rebuilt after the models were trained, so this is an estimate of the overlap and not an exact audit of the training run.

In [ ]:
# Paths
train_dir = f'{DRIVE}/NIH_processed/train'
test_dir = f'{DRIVE}/NIH_processed/test'
entry_csv = find_file('Data_Entry_2017_v2020*.csv')

# Image index to patient identifier
patient_of = pd.read_csv(entry_csv).set_index('Image Index')['Patient ID']


def list_images(folder):
    return [f for r, _, fs in os.walk(folder) for f in fs if f.lower().endswith('.png')]


def source_name(f):
    # Augmented files carry an aug_<n>_ prefix in front of the original file name
    return re.sub(r'^aug_\d+_', '', f)


train_files, test_files = list_images(train_dir), list_images(test_dir)
train_patients = {f: patient_of[source_name(f)] for f in train_files if source_name(f) in patient_of.index}
test_patients = {f: patient_of[source_name(f)] for f in test_files if source_name(f) in patient_of.index}

shared = set(train_patients.values()) & set(test_patients.values())
test_from_shared = sum(p in shared for p in test_patients.values())

augmented = [f for f in train_files if f.startswith('aug_')]
test_sources = {source_name(f) for f in test_files}
augmented_from_test = sum(source_name(f) in test_sources for f in augmented)

print(f'Training files: {len(train_files)}, test files: {len(test_files)}')
print(f'Patients with images in both folders: {len(shared)}')
print(f'Test images from those patients: {test_from_shared} ({100 * test_from_shared / len(test_files):.1f}%)')
print(f'Augmented training files: {len(augmented)}, derived from a test image: {augmented_from_test}')

Training files: 5320, test files: 1784
Patients with images in both folders: 543
Test images from those patients: 711 (39.9%)
Augmented training files: 332, derived from a test image: 123


## **2. 95% confidence intervals for the deployment-selected models**

Sensitivity, specificity and accuracy use Wilson score intervals. The interval for balanced accuracy is obtained by simulation from Beta posteriors for sensitivity and specificity (Jeffreys prior). The interval for AUROC uses the Hanley and McNeil approximation. All three treat images as independent, so they do not allow for several images per patient and are likely to be narrower than the true uncertainty.

The counts are the confusion matrices printed by the evaluation cells of the main notebooks and by sections 4 and 5. The NIH EfficientNetB4 INT8 row is the dynamic-range model; its counts are reconstructed from the reported sensitivity (72.66%) and specificity (84.28%).

In [ ]:
from math import sqrt


def wilson(k, n, z=1.96):
    p = k / n
    denom = 1 + z * z / n
    centre = p + z * z / (2 * n)
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return 100 * (centre - half) / denom, 100 * (centre + half) / denom


def balanced_accuracy_interval(tp, fn, tn, fp, draws=200000, seed=0):
    rng = np.random.default_rng(seed)
    sens = rng.beta(tp + 0.5, fn + 0.5, draws)
    spec = rng.beta(tn + 0.5, fp + 0.5, draws)
    lo, hi = np.percentile((sens + spec) / 2, [2.5, 97.5])
    return 100 * lo, 100 * hi


def hanley_mcneil(auc, n_pos, n_neg, z=1.96):
    q1 = auc / (2 - auc)
    q2 = 2 * auc * auc / (1 + auc)
    se = sqrt((auc * (1 - auc) + (n_pos - 1) * (q1 - auc ** 2) + (n_neg - 1) * (q2 - auc ** 2)) / (n_pos * n_neg))
    return auc - z * se, auc + z * se


# Confusion matrix counts (TP, FN, TN, FP) and AUROC
models = {
    'NIH EfficientNetB4, original':            (404, 108, 1132, 140, 0.8986),
    'NIH EfficientNetB4, INT8 (dynamic range)': (372, 140, 1072, 200, 0.8563),
    'Mendeley Xception, original':             (378, 12, 227, 7, 0.9933),
    'Mendeley Xception, INT8':                 (382, 8, 224, 10, 0.9934),
}

rows = []
for name, (tp, fn, tn, fp, auc) in models.items():
    n = tp + fn + tn + fp
    fmt = lambda v, ci: f'{v:.2f} ({ci[0]:.2f} to {ci[1]:.2f})'
    lo, hi = hanley_mcneil(auc, tp + fn, tn + fp)
    rows.append({
        'Model': name,
        'Accuracy (%)': fmt(100 * (tp + tn) / n, wilson(tp + tn, n)),
        'Sensitivity (%)': fmt(100 * tp / (tp + fn), wilson(tp, tp + fn)),
        'Specificity (%)': fmt(100 * tn / (tn + fp), wilson(tn, tn + fp)),
        'Balanced accuracy (%)': fmt(50 * (tp / (tp + fn) + tn / (tn + fp)), balanced_accuracy_interval(tp, fn, tn, fp)),
        'AUROC': f'{auc:.3f} ({lo:.3f} to {hi:.3f})',
    })
pd.set_option('display.width', 250, 'display.max_columns', None, 'display.max_colwidth', None)
pd.DataFrame(rows).set_index('Model')

,Accuracy (%),Sensitivity (%),Specificity (%),Balanced accuracy (%),AUROC
Model,,,,,
"NIH EfficientNetB4, original",86.10 (84.42 to 87.63),78.91 (75.16 to 82.22),88.99 (87.15 to 90.60),83.95 (81.91 to 85.83),0.899 (0.880 to 0.917)
"NIH EfficientNetB4, INT8 (dynamic range)",80.94 (79.05 to 82.70),72.66 (68.64 to 76.34),84.28 (82.17 to 86.17),78.47 (76.23 to 80.56),0.856 (0.835 to 0.878)
"Mendeley Xception, original",96.96 (95.29 to 98.04),96.92 (94.70 to 98.23),97.01 (93.95 to 98.54),96.97 (95.24 to 98.06),0.993 (0.988 to 0.999)
"Mendeley Xception, INT8",97.12 (95.49 to 98.17),97.95 (96.01 to 98.96),95.73 (92.31 to 97.66),96.84 (95.02 to 98.00),0.993 (0.988 to 0.999)


## **3. Multi-label split reconciliation**

Recomputes the image and patient counts of the multi-label experiment from the label files: the full collection, the held-out portion, the pool before multi-label random over-sampling (ML-ROS), the over-sampled training set, and the patients shared between the validation and test halves.

In [ ]:
# Files
train_balanced = pd.read_csv(find_file('train_balanced_all.csv'))
held_out = pd.read_excel(find_file('val_original.xlsx'))
entries = pd.read_csv(find_file('Data_Entry_2017_v2020*.csv'))

print(f"Full collection: {len(entries)} images, {entries['Patient ID'].nunique()} patients")
print(f"Held-out portion: {len(held_out)} images, {held_out['Patient ID'].nunique()} patients")

# Pool available for training before over-sampling
pool = entries[~entries['Image Index'].isin(held_out['Image Index'])]
print(f"Pool before ML-ROS: {len(pool)} images, {pool['Patient ID'].nunique()} patients")

print(f"ML-ROS training set: {len(train_balanced)} rows, {train_balanced['Image Index'].nunique()} distinct images, "
      f"{train_balanced['Patient ID'].nunique()} patients")
print(f"No Finding: {(pool['Finding Labels'] == 'No Finding').sum()} in the pool, "
      f"{(train_balanced['Finding Labels'] == 'No Finding').sum()} in the ML-ROS training set")
print(f"Patients in both the pool and the held-out portion: "
      f"{len(set(pool['Patient ID']) & set(held_out['Patient ID']))}")

# Validation and test halves of the held-out portion
half_a = held_out.sample(frac=0.5, random_state=123)
half_b = held_out.drop(half_a.index)
print(f"Validation and test halves: {len(half_a)} and {len(half_b)} images, "
      f"{len(set(half_a['Patient ID']) & set(half_b['Patient ID']))} patients in both")

Full collection: 112120 images, 30805 patients
Held-out portion: 22294 images, 6161 patients
Pool before ML-ROS: 89826 images, 24644 patients
ML-ROS training set: 61446 rows, 51113 distinct images, 14885 patients
No Finding: 48373 in the pool, 9660 in the ML-ROS training set
Patients in both the pool and the held-out portion: 0
Validation and test halves: 11147 and 11147 images, 2089 patients in both


## **4. FP32 TFLite baselines: size and timing**

Each quantised model is compared with an unquantised TFLite version of the same network, so that the effect of quantisation is separated from the change of runtime from Keras to TFLite. Sizes are reported for the saved Keras file, the FP32 weights implied by the parameter count (4 bytes each), the FP32 TFLite file and the quantised file.

Timing uses the TFLite interpreter, 50 test images (25 per class), 5 warm-up calls, and three rounds in which the two files are timed in alternating order. Pre-processing is not included. Times are multiplied by the clock adjustment factor.

In [ ]:
prep_efficientnet = tf.keras.applications.efficientnet.preprocess_input
prep_xception = tf.keras.applications.xception.preprocess_input

jobs = {
    'NIH EfficientNetB4': dict(
        keras=f'{DRIVE}/NIH_models/Results/EffNetB4_epoch_15.h5',
        fp32=f'{DRIVE}/FP32_TFLite_check/NIH_EfficientNetB4_fp32.tflite',
        quantised=f'{DRIVE}/NIH_models/Quantized/EffNetB4_quantized_hybrid.tflite',
        kind='dynamic range', images=f'{DRIVE}/NIH_processed/test', size=(224, 224), prep=prep_efficientnet),
    'Mendeley Xception': dict(
        keras=f'{DRIVE}/chest_xray/base_model_3.keras',
        fp32=f'{DRIVE}/FP32_TFLite_check/Mendeley_Xception_fp32.tflite',
        quantised=f'{DRIVE}/chest_xray/PTQ_Quantized_Ensemble/base_model_3_int8.tflite',
        kind='INT8', images=f'{DRIVE}/chest_xray/test', size=(299, 299), prep=prep_xception),
}
os.makedirs(f'{DRIVE}/FP32_TFLite_check', exist_ok=True)
mb = lambda path: os.path.getsize(path) / 1024 ** 2
from PIL import Image

for name, j in jobs.items():
    model = tf.keras.models.load_model(j['keras'], compile=False)
    params = model.count_params()
    if not os.path.exists(j['fp32']):
        with open(j['fp32'], 'wb') as f:
            f.write(tf.lite.TFLiteConverter.from_keras_model(model).convert())
    print(f'\n{name}')
    print(f"  parameters {params / 1e6:.2f} M | FP32 weights {params * 4 / 1024 ** 2:.2f} MB | saved Keras file {mb(j['keras']):.2f} MB")
    print(f"  FP32 TFLite {mb(j['fp32']):.2f} MB | {j['kind']} TFLite {mb(j['quantised']):.2f} MB "
          f"({100 * (1 - mb(j['quantised']) / mb(j['fp32'])):.1f}% smaller than FP32 TFLite, "
          f"{100 * (1 - mb(j['quantised']) / (params * 4 / 1024 ** 2)):.1f}% smaller than the FP32 weights)")

    # 25 images per class
    paths = [p for sub in sorted(os.listdir(j['images'])) for p in sorted(glob.glob(f"{j['images']}/{sub}/*.*"))[:25]]
    batch = [j['prep'](np.array(Image.open(p).convert('RGB').resize(j['size']), dtype=np.float32)[None]) for p in paths]

    interpreters = {}
    for label, path in (('FP32 TFLite', j['fp32']), (j['kind'], j['quantised'])):
        it = tf.lite.Interpreter(model_path=path)
        it.allocate_tensors()
        interpreters[label] = it

    def time_images(it):
        d_in, d_out = it.get_input_details()[0], it.get_output_details()[0]
        for _ in range(5):
            it.set_tensor(d_in['index'], batch[0]); it.invoke()
        out = []
        for x in batch:
            it.set_tensor(d_in['index'], x)
            t0 = time.perf_counter(); it.invoke()
            out.append((time.perf_counter() - t0) * 1000 * clock_factor)
        return out

    times = {k: [] for k in interpreters}
    for r in range(3):
        order = list(interpreters) if r % 2 == 0 else list(interpreters)[::-1]
        for k in order:
            times[k] += time_images(interpreters[k])
    for k, t in times.items():
        print(f'  {k}: mean {np.mean(t):.1f} ms, SD {np.std(t):.1f} ms, median {np.median(t):.1f} ms ({len(t)} timings)')
    keys = list(times)
    print(f'  {keys[1]} against FP32 TFLite: {100 * (np.mean(times[keys[1]]) / np.mean(times[keys[0]]) - 1):+.1f}% in mean time')
    del model
    tf.keras.backend.clear_session()


NIH EfficientNetB4
  parameters 18.13 M | FP32 weights 69.17 MB | saved Keras file 208.32 MB
  FP32 TFLite 68.63 MB | dynamic range TFLite 18.59 MB (72.9% smaller than FP32 TFLite, 73.1% smaller than the FP32 weights)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  FP32 TFLite: mean 220.9 ms, SD 56.6 ms, median 193.9 ms (150 timings)
  dynamic range: mean 448.8 ms, SD 402.5 ms, median 293.2 ms (150 timings)
  dynamic range against FP32 TFLite: +103.2% in mean time

Mendeley Xception
  parameters 21.39 M | FP32 weights 81.58 MB | saved Keras file 82.06 MB
  FP32 TFLite 81.33 MB | INT8 TFLite 21.61 MB (73.4% smaller than FP32 TFLite, 73.5% smaller than the FP32 weights)
  FP32 TFLite: mean 565.8 ms, SD 150.8 ms, median 491.4 ms (150 timings)
  INT8: mean 595.1 ms, SD 117.2 ms, median 530.5 ms (150 timings)
  INT8 against FP32 TFLite: +5.2% in mean time


## **5. Xception, original against INT8: paired comparison**

The original Keras Xception and the INT8 TFLite model are evaluated on the same 624 Mendeley test images. Agreement between the two is assessed with an exact McNemar test on the discordant images, and the differences in accuracy, balanced accuracy and AUROC are given with paired bootstrap intervals (5,000 resamples of the test images).

In [ ]:
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator

chest = f'{DRIVE}/chest_xray'
int8_path = f'{chest}/PTQ_Quantized_Ensemble/base_model_3_int8.tflite'
model = tf.keras.models.load_model(f'{chest}/base_model_3.keras', compile=False)
size = model.input_shape[1:3]


def test_generator():
    return ImageDataGenerator(preprocessing_function=prep_xception).flow_from_directory(
        f'{chest}/test', target_size=size, class_mode='categorical', batch_size=1, shuffle=False)


def p_pneumonia(output):
    output = np.asarray(output).reshape(-1)
    return float(output[1]) if output.size == 2 else float(output[0])


def accuracy(y, p):
    return (((p >= 0.5).astype(int)) == y).mean()


def balanced_accuracy(y, p):
    pred = (p >= 0.5).astype(int)
    return 0.5 * ((pred[y == 1] == 1).mean() + (pred[y == 0] == 0).mean())


# Original model
gen = test_generator()
y = gen.classes.copy()
n = len(y)
p_keras = np.array([p_pneumonia(model.predict(next(gen)[0], verbose=0)) for _ in range(n)])

# INT8 model
interp = tf.lite.Interpreter(model_path=int8_path)
interp.allocate_tensors()
d_in, d_out = interp.get_input_details()[0], interp.get_output_details()[0]
gen = test_generator()
p_int8 = []
for _ in range(n):
    interp.set_tensor(d_in['index'], next(gen)[0].astype(np.float32))
    interp.invoke()
    p_int8.append(p_pneumonia(interp.get_tensor(d_out['index'])))
p_int8 = np.array(p_int8)

for label, p in (('Original (Keras)', p_keras), ('INT8 (TFLite)', p_int8)):
    tn, fp, fn, tp = confusion_matrix(y, (p >= 0.5).astype(int)).ravel()
    print(f'{label}: TN {tn} FP {fp} FN {fn} TP {tp} | accuracy {accuracy(y, p):.4f}, sensitivity {tp / (tp + fn):.4f}, '
          f'specificity {tn / (tn + fp):.4f}, balanced accuracy {balanced_accuracy(y, p):.4f}, AUROC {roc_auc_score(y, p):.4f}')

# Discordant images and exact McNemar test
right_keras = (p_keras >= 0.5).astype(int) == y
right_int8 = (p_int8 >= 0.5).astype(int) == y
b, c = int((right_keras & ~right_int8).sum()), int((~right_keras & right_int8).sum())
print(f'\nOriginal correct and INT8 wrong: {b}; original wrong and INT8 correct: {c}; '
      f'McNemar exact p = {binomtest(b, b + c, 0.5).pvalue:.4f}')

# Paired bootstrap of the differences (INT8 minus original)
rng = np.random.default_rng(0)
diffs = {'accuracy': [], 'balanced accuracy': [], 'AUROC': []}
for _ in range(5000):
    idx = rng.integers(0, n, n)
    yy = y[idx]
    if yy.min() == yy.max():
        continue
    diffs['accuracy'].append(accuracy(yy, p_int8[idx]) - accuracy(yy, p_keras[idx]))
    diffs['balanced accuracy'].append(balanced_accuracy(yy, p_int8[idx]) - balanced_accuracy(yy, p_keras[idx]))
    diffs['AUROC'].append(roc_auc_score(yy, p_int8[idx]) - roc_auc_score(yy, p_keras[idx]))
for k, v in diffs.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f'Difference in {k}: {np.mean(v):+.4f} (95% CI {lo:+.4f} to {hi:+.4f})')
del model
tf.keras.backend.clear_session()

Found 624 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Original (Keras): TN 227 FP 7 FN 12 TP 378 | accuracy 0.9696, sensitivity 0.9692, specificity 0.9701, balanced accuracy 0.9697, AUROC 0.9933
INT8 (TFLite): TN 224 FP 10 FN 8 TP 382 | accuracy 0.9712, sensitivity 0.9795, specificity 0.9573, balanced accuracy 0.9684, AUROC 0.9934

Original correct and INT8 wrong: 3; original wrong and INT8 correct: 4; McNemar exact p = 1.0000
Difference in accuracy: +0.0015 (95% CI -0.0064 to +0.0096)
Difference in balanced accuracy: -0.0014 (95% CI -0.0109 to +0.0072)
Difference in AUROC: +0.0000 (95% CI -0.0014 to +0.0016)


## **6. Grad-CAM speed benchmark**

Grad-CAM is timed under three conditions on 50 test images (25 per class): the eager computation at full input size, the same computation compiled with `tf.function`, and the compiled computation on an input reduced to 128 x 128 pixels with the map resized back to full size. The reduced input is tried explicitly. If a network does not accept it, the cell says so and no result is reported for that condition, so a speed-up is never credited to a reduction that was not applied. For the conditions that run, the map is compared with the eager full-size map by Pearson correlation and by intersection over union at a threshold of 0.5. Maps are from the original Keras models, as TFLite does not expose gradients. Times are multiplied by the clock adjustment factor.

In [ ]:
# Grad-CAM speed benchmark (self-contained: only needs Google Drive to be available)
import os
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '4')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '1')
import glob, time
import numpy as np, psutil, tensorflow as tf
from PIL import Image

DRIVE = '/content/drive/MyDrive'
if not os.path.exists(DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

# Clock adjustment used throughout the study
clock_factor = (psutil.cpu_freq().current / 1000) / 1.5
print(f'Clock adjustment factor: {clock_factor:.2f}')

prep_efficientnet = tf.keras.applications.efficientnet.preprocess_input
prep_xception = tf.keras.applications.xception.preprocess_input

gradcam_jobs = {
    'NIH EfficientNetB4': dict(model=f'{DRIVE}/NIH_models/Results/EffNetB4_epoch_15.h5',
                               images=f'{DRIVE}/NIH_processed/test', size=224, prep=prep_efficientnet),
    'Mendeley Xception': dict(model=f'{DRIVE}/chest_xray/base_model_3.keras',
                              images=f'{DRIVE}/chest_xray/test', size=299, prep=prep_xception),
}
reduced = 128


def single(x):
    # Some Keras versions return a one-element list for a single tensor
    return x[0] if isinstance(x, (list, tuple)) else x


def iou(a, b, threshold=0.5):
    a, b = a >= threshold, b >= threshold
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


for name, j in gradcam_jobs.items():
    model = tf.keras.models.load_model(j['model'], compile=False)
    size = j['size']

    # Last top-level layer with a four-dimensional output (taken from the outer graph, so nested models also work),
    # and the first model output
    def layer_output(layer):
        try:
            return single(layer._inbound_nodes[0].output_tensors)
        except Exception:
            return single(layer.output)

    conv_layer, conv_tensor = None, None
    for layer in model.layers:
        try:
            t = layer_output(layer)
            if len(t.shape) == 4:
                conv_layer, conv_tensor = layer.name, t
        except Exception:
            pass
    output = single(model.outputs)
    grad_model = tf.keras.Model(model.inputs, [conv_tensor, output])
    class_index = 1 if output.shape[-1] == 2 else 0

    def heatmap(x):
        with tf.GradientTape() as tape:
            tape.watch(x)
            conv, preds = grad_model(x)
            conv, preds = single(conv), single(preds)
            score = preds[:, class_index]
        grads = tape.gradient(score, conv)
        weights = tf.reduce_mean(grads, axis=(0, 1, 2))
        cam = tf.nn.relu(tf.tensordot(conv[0], weights, axes=[[2], [0]]))
        return tf.image.resize(cam[..., tf.newaxis], (size, size))[..., 0]

    compiled = tf.function(heatmap)

    def normalise(cam):
        cam = cam.numpy()
        return (cam - cam.min()) / (cam.max() - cam.min()) if cam.max() > cam.min() else np.zeros_like(cam)

    # 25 images per class
    paths = [p for sub in sorted(os.listdir(j['images'])) for p in sorted(glob.glob(f"{j['images']}/{sub}/*.*"))[:25]]
    images = [tf.constant(j['prep'](np.array(Image.open(p).convert('RGB').resize((size, size)), dtype=np.float32)[None]))
              for p in paths]

    # Warm-up, and a check that the reduced input is accepted
    heatmap(images[0]); compiled(images[0])
    try:
        compiled(tf.image.resize(images[0], (reduced, reduced)))
        reduced_ok = True
    except Exception as e:
        reduced_ok = False
        print(f'{name}: reduced {reduced} x {reduced} input not accepted ({type(e).__name__})')

    t_eager, t_compiled, t_reduced, corr_c, iou_c, corr_r, iou_r = [], [], [], [], [], [], []
    for x in images:
        t0 = time.perf_counter(); ref = normalise(heatmap(x)); t_eager.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); cam_c = normalise(compiled(x)); t_compiled.append(time.perf_counter() - t0)
        corr_c.append(np.corrcoef(ref.ravel(), cam_c.ravel())[0, 1]); iou_c.append(iou(ref, cam_c))
        if reduced_ok:
            t0 = time.perf_counter()
            cam_r = normalise(compiled(tf.image.resize(x, (reduced, reduced), method='bilinear')))
            t_reduced.append(time.perf_counter() - t0)
            corr_r.append(np.corrcoef(ref.ravel(), cam_r.ravel())[0, 1]); iou_r.append(iou(ref, cam_r))

    s = lambda t: f'{np.mean(t) * clock_factor:.3f} +/- {np.std(t) * clock_factor:.3f} s'
    print(f'\n{name} ({len(images)} images, layer {conv_layer}, class index {class_index})')
    print(f'  Eager, full size:      {s(t_eager)}')
    print(f'  Compiled, full size:   {s(t_compiled)} | speed-up {np.mean(t_eager) / np.mean(t_compiled):.2f}x | '
          f'correlation {np.mean(corr_c):.3f}, IoU {np.mean(iou_c):.3f}')
    if reduced_ok:
        print(f'  Compiled, {reduced} x {reduced}: {s(t_reduced)} | speed-up {np.mean(t_eager) / np.mean(t_reduced):.2f}x | '
              f'correlation {np.mean(corr_r):.3f}, IoU {np.mean(iou_r):.3f}')
    del model, grad_model
    tf.keras.backend.clear_session()

Clock adjustment factor: 1.47
NIH EfficientNetB4: reduced 128 x 128 input not accepted (ValueError)

NIH EfficientNetB4 (50 images, layer top_activation, class index 0)
  Eager, full size:      4.287 +/- 0.957 s
  Compiled, full size:   0.344 +/- 0.153 s | speed-up 12.47x | correlation 1.000, IoU 1.000
Mendeley Xception: reduced 128 x 128 input not accepted (ValueError)


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]



Mendeley Xception (50 images, layer block14_sepconv2_act, class index 1)
  Eager, full size:      3.288 +/- 0.649 s
  Compiled, full size:   0.674 +/- 0.195 s | speed-up 4.88x | correlation nan, IoU 0.980


### **Ensemble: comparison of two input pipelines**
 - Ensemble: comparison of two input pipelines
 - The original ensemble cells read the test images through a single generator that applies the ResNet50 pre-processing, resize each batch to the input size of every base model, and then apply that model's own pre-processing.

- The cells that evaluated the INT8 base models read the images through one generator per model.

- This section runs the original pipeline for the Keras and INT8 base models and compares the per-model accuracies.


In [ ]:
import os, sys, subprocess, glob
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna']); import optuna
import numpy as np, tensorflow as tf
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BASE = '/content/drive/MyDrive/chest_xray'
TRAIN, TEST = f'{BASE}/train', f'{BASE}/test'
QDIR = f'{BASE}/PTQ_Quantized_Ensemble'
OUT = f'{BASE}/ensemble_validation_tuned'
PRE = [tf.keras.applications.resnet.preprocess_input, tf.keras.applications.densenet.preprocess_input,
       tf.keras.applications.inception_v3.preprocess_input, tf.keras.applications.xception.preprocess_input,
       tf.keras.applications.mobilenet_v2.preprocess_input]
NAMES = ['ResNet50', 'DenseNet121', 'InceptionV3', 'Xception', 'MobileNetV2']

models = [tf.keras.models.load_model(f'{BASE}/base_model_{i}.keras', compile=False) for i in range(5)]
meta = tf.keras.models.load_model(f'{BASE}/nuclear_ensemble_meta_learner.keras', compile=False)
saved_file = [f for f in glob.glob(f'{BASE}/*.npy') if 'optimal_weights' in f.lower() and 'student' not in f.lower()][0]
w_saved = np.load(saved_file)

# Single generator with the ResNet50 pre-processing at 224 x 224 (as in the original ensemble cells)
def shared_generator(path, subset=None):
    gen = ImageDataGenerator(preprocessing_function=PRE[0], validation_split=0.2 if subset else 0.0)
    kw = dict(subset=subset, seed=123) if subset else {}
    return gen.flow_from_directory(path, target_size=(224, 224), class_mode='categorical',
                                   batch_size=32, shuffle=False, **kw)

def keras_features(it):
    out = [[] for _ in models]
    for _ in range(len(it)):
        x, _ = next(it)
        for i, (m, pf) in enumerate(zip(models, PRE)):
            h, w = m.input_shape[1:3]
            out[i].append(m.predict(pf(tf.image.resize(x, (h, w))), verbose=0))
    return np.stack([np.vstack(o) for o in out], axis=1), it.classes

def int8_features():
    interps = []
    for i in range(5):
        it = tf.lite.Interpreter(model_path=f'{QDIR}/base_model_{i}_int8.tflite'); it.allocate_tensors()
        interps.append((it, it.get_input_details()[0], it.get_output_details()[0]))
    gen = shared_generator(TEST); out = [[] for _ in interps]
    for _ in range(len(gen)):
        x, _ = next(gen)
        for j in range(len(x)):
            for i, ((it, d_in, d_out), pf) in enumerate(zip(interps, PRE)):
                h, w = [int(v) for v in d_in['shape'][1:3]]
                inp = pf(tf.image.resize(x[j:j + 1], (h, w))).numpy().astype(np.float32)
                it.set_tensor(d_in['index'], inp); it.invoke()
                out[i].append(it.get_tensor(d_out['index'])[0])
    return np.stack([np.array(o) for o in out], axis=1)

Xv, yv = keras_features(shared_generator(TRAIN, 'validation'))
Xt, yt = keras_features(shared_generator(TEST))
Xq = int8_features()
print('validation', len(yv), '| test', len(yt))

wavg = lambda X, w: np.tensordot(w, X, axes=([0], [1]))
blend = lambda X, w: 0.6 * meta.predict(X, verbose=0) + 0.4 * wavg(X, w)
def show(name, p, y=None):
    y = yt if y is None else y
    pred = p.argmax(1); tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    se, sp = tp / (tp + fn), tn / (tn + fp)
    print(f'{name:46s} TN {tn} FP {fp} FN {fn} TP {tp} | acc {(tp + tn) / len(y):.4f} sens {se:.4f} spec {sp:.4f} '
          f'bal {(se + sp) / 2:.4f} AUROC {roc_auc_score(y, p[:, 1]):.4f}')

print('\nPer-model test results, original pipeline (archived training logs: ResNet50 0.9375, DenseNet121 0.9087, '
      'InceptionV3 0.9359, Xception 0.9696, MobileNetV2 0.9183; archived INT8: 0.9199, 0.6250, 0.9247, 0.9712, 0.9263)')
for i, n in enumerate(NAMES):
    show(f'{n} Keras', Xt[:, i]); show(f'{n} INT8', Xq[:, i])

print('\nOriginal Keras ensemble, blend = 0.6 x meta-learner + 0.4 x weighted base models')
show('saved weights (archived result: 96.79 / 0.9915)', blend(Xt, w_saved))

def objective(trial):
    w = np.array([trial.suggest_float(f'w{i}', 0, 1) for i in range(5)]); w /= w.sum()
    return (wavg(Xv, w).argmax(1) == yv).mean()
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100)
w_val = np.array([study.best_params[f'w{i}'] for i in range(5)]); w_val /= w_val.sum()
print('validation-tuned weights (this pipeline):', dict(zip(NAMES, np.round(w_val, 4))), '| validation accuracy', round(study.best_value, 4))
show('equal weights', blend(Xt, np.ones(5) / 5))
show('validation-tuned weights', blend(Xt, w_val))
print('\nINT8 ensemble, same pipeline, same meta-learner')
show('INT8, validation-tuned weights', blend(Xq, w_val))
show('INT8, saved weights', blend(Xq, w_saved))

po, pq = blend(Xt, w_val), blend(Xq, w_val)
ok_o, ok_q = po.argmax(1) == yt, pq.argmax(1) == yt
b, c = int((ok_o & ~ok_q).sum()), int((~ok_o & ok_q).sum())
print(f'\ndiscordant pairs: original right/INT8 wrong {b}, original wrong/INT8 right {c}; McNemar exact p = '
      f'{binomtest(b, b + c, 0.5).pvalue:.4f}' if b + c else '\nno discordant pairs')
np.savez(f'{OUT}/ensemble_predictions_original_pipeline.npz', yt=yt, yv=yv, Xt=Xt, Xv=Xv, Xq=Xq, w_val=w_val, w_saved=w_saved)

Found 1045 images belonging to 2 classes.


Found 624 images belonging to 2 classes.


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Found 624 images belonging to 2 classes.
validation 1045 | test 624

Per-model test results, original pipeline (archived training logs: ResNet50 0.9375, DenseNet121 0.9087, InceptionV3 0.9359, Xception 0.9696, MobileNetV2 0.9183; archived INT8: 0.9199, 0.6250, 0.9247, 0.9712, 0.9263)
ResNet50 Keras                                 TN 217 FP 17 FN 22 TP 368 | acc 0.9375 sens 0.9436 spec 0.9274 bal 0.9355 AUROC 0.9839
ResNet50 INT8                                  TN 214 FP 20 FN 24 TP 366 | acc 0.9295 sens 0.9385 spec 0.9145 bal 0.9265 AUROC 0.9815
DenseNet121 Keras                              TN 221 FP 13 FN 35 TP 355 | acc 0.9231 sens 0.9103 spec 0.9444 bal 0.9274 AUROC 0.9842
DenseNet121 INT8                               TN 0 FP 234 FN 0 TP 390 | acc 0.6250 sens 1.0000 spec 0.0000 bal 0.5000 AUROC 0.6130
InceptionV3 Keras                              TN 185 FP 49 FN 37 TP 353 | acc 0.8622 sens 0.9051 spec 0.7906 bal 0.8479 AUROC 0.9268
InceptionV3 INT8                               

### **Test folder identity check**

In [ ]:
# NIH test folder identity check
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, roc_auc_score

DRIVE = '/content/drive/MyDrive'
if not os.path.exists(DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

model = tf.keras.models.load_model(f'{DRIVE}/NIH_models/Results/EffNetB4_epoch_15.h5', compile=False)
test = tf.keras.utils.image_dataset_from_directory(f'{DRIVE}/NIH_processed/test', label_mode='binary',
                                                   image_size=(224, 224), batch_size=32, shuffle=False)
y = np.concatenate([b.numpy().ravel() for _, b in test]).astype(int)
p = model.predict(test, verbose=0).ravel()
tn, fp, fn, tp = confusion_matrix(y, p >= 0.5).ravel()
print(f'This folder: TN {tn} FP {fp} FN {fn} TP {tp} | accuracy {(tp + tn) / len(y):.4f} | AUROC {roc_auc_score(y, p):.4f}')
print('Archived:    TN 1132 FP 140 FN 108 TP 404 | accuracy 0.8610 | AUROC 0.8986')

Found 1784 files belonging to 2 classes.
This folder: TN 1122 FP 150 FN 58 TP 454 | accuracy 0.8834 | AUROC 0.9401
Archived:    TN 1132 FP 140 FN 108 TP 404 | accuracy 0.8610 | AUROC 0.8986


### **Grad-CAM at a true 128 × 128 input**

In [ ]:
# Grad-CAM at reduced input size (128 x 128), with the saved weights placed in a network built for that size
import os
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '4')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '1')
import glob, time
import numpy as np, psutil, tensorflow as tf
from PIL import Image

DRIVE = '/content/drive/MyDrive'
if not os.path.exists(DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

clock_factor = (psutil.cpu_freq().current / 1000) / 1.5
print(f'Clock adjustment factor: {clock_factor:.2f}')

jobs = {
    'NIH EfficientNetB4': dict(model=f'{DRIVE}/NIH_models/Results/EffNetB4_epoch_15.h5',
                               images=f'{DRIVE}/NIH_processed/test', size=224,
                               prep=tf.keras.applications.efficientnet.preprocess_input),
    'Mendeley Xception': dict(model=f'{DRIVE}/chest_xray/base_model_3.keras',
                              images=f'{DRIVE}/chest_xray/test', size=299,
                              prep=tf.keras.applications.xception.preprocess_input),
}
reduced = 128


def single(x):
    return x[0] if isinstance(x, (list, tuple)) else x


def rebuild_for_size(model, size):
    # Same layers and the same saved weights, with the input layer(s) declared at size x size
    cfg = model.get_config()
    def patch(o):
        if isinstance(o, dict):
            for k in ('batch_shape', 'batch_input_shape'):
                if k in o and isinstance(o[k], (list, tuple)) and len(o[k]) == 4:
                    o[k] = [None, size, size, o[k][3]]
            for v in o.values():
                patch(v)
        elif isinstance(o, list):
            for v in o:
                patch(v)
    patch(cfg)
    new = type(model).from_config(cfg)
    new.set_weights(model.get_weights())
    return new


def make_grad_model(model):
    if isinstance(model, tf.keras.Sequential):
        # A Sequential model is chained layer by layer so that the last convolutional output is connected to the prediction
        inp = tf.keras.Input(shape=model.input_shape[1:])
        x, conv, conv_layer = inp, None, None
        for layer in model.layers:
            x = layer(x)
            if len(single(x).shape) == 4:
                conv, conv_layer = single(x), layer.name
        x = single(x)
        return tf.keras.Model(inp, [conv, x]), conv_layer, (1 if x.shape[-1] == 2 else 0)
    def layer_output(layer):
        try:
            return single(layer._inbound_nodes[0].output_tensors)
        except Exception:
            return single(layer.output)
    conv_layer, conv_tensor = None, None
    for layer in model.layers:
        try:
            t = layer_output(layer)
            if len(t.shape) == 4:
                conv_layer, conv_tensor = layer.name, t
        except Exception:
            pass
    output = single(model.outputs)
    return tf.keras.Model(model.inputs, [conv_tensor, output]), conv_layer, (1 if output.shape[-1] == 2 else 0)


def make_heatmap(grad_model, class_index, out_size):
    def heatmap(x):
        with tf.GradientTape() as tape:
            tape.watch(x)
            conv, preds = grad_model(x)
            conv, preds = single(conv), single(preds)
            score = preds[:, class_index]
        grads = tape.gradient(score, conv)
        weights = tf.reduce_mean(grads, axis=(0, 1, 2))
        cam = tf.nn.relu(tf.tensordot(conv[0], weights, axes=[[2], [0]]))
        return tf.image.resize(cam[..., tf.newaxis], (out_size, out_size))[..., 0]
    return heatmap


def normalise(cam):
    cam = cam.numpy()
    return (cam - cam.min()) / (cam.max() - cam.min()) if cam.max() > cam.min() else np.zeros_like(cam)


def iou(a, b, threshold=0.5):
    a, b = a >= threshold, b >= threshold
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else np.nan


def correlation(a, b):
    if a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a.ravel(), b.ravel())[0, 1])


for name, j in jobs.items():
    size = j['size']
    model = tf.keras.models.load_model(j['model'], compile=False)
    grad_full, conv_layer, class_index = make_grad_model(model)
    small = rebuild_for_size(model, reduced)
    grad_small, _, _ = make_grad_model(small)

    # The saved network against a reduced input, and the rebuilt network against the saved one at full size
    try:
        model(tf.zeros((1, reduced, reduced, 3)))
        print(f'{name}: the saved network accepts {reduced} x {reduced} directly')
    except Exception as e:
        print(f'{name}: the saved network does not accept {reduced} x {reduced} directly ({type(e).__name__})')
    check = rebuild_for_size(model, size)
    probe = tf.constant(j['prep'](np.random.RandomState(0).rand(1, size, size, 3).astype(np.float32) * 255))
    print(f'{name}: largest output difference, rebuilt network against saved network at {size} x {size}: '
          f'{float(np.abs(single(model(probe)).numpy() - single(check(probe)).numpy()).max()):.2e}')
    del check

    eager = make_heatmap(grad_full, class_index, size)
    compiled = tf.function(make_heatmap(grad_full, class_index, size))
    compiled_small = tf.function(make_heatmap(grad_small, class_index, size))

    # 25 images per class
    paths = [p for sub in sorted(os.listdir(j['images'])) for p in sorted(glob.glob(f"{j['images']}/{sub}/*.*"))[:25]]
    images = [tf.constant(j['prep'](np.array(Image.open(p).convert('RGB').resize((size, size)), dtype=np.float32)[None]))
              for p in paths]
    smalls = [tf.image.resize(x, (reduced, reduced), method='bilinear') for x in images]

    eager(images[0]); compiled(images[0]); compiled_small(smalls[0])      # warm-up

    t_e, t_c, t_s, c_c, i_c, c_s, i_s = [], [], [], [], [], [], []
    flat_e = flat_s = 0
    for x, xs in zip(images, smalls):
        t0 = time.perf_counter(); ref = normalise(eager(x)); t_e.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); cam_c = normalise(compiled(x)); t_c.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); cam_s = normalise(compiled_small(xs)); t_s.append(time.perf_counter() - t0)
        flat_e += int(not ref.any()); flat_s += int(not cam_s.any())
        c_c.append(correlation(ref, cam_c)); i_c.append(iou(ref, cam_c))
        c_s.append(correlation(ref, cam_s)); i_s.append(iou(ref, cam_s))

    s = lambda t: f'{np.mean(t) * clock_factor:.3f} +/- {np.std(t) * clock_factor:.3f} s'
    n_ok = lambda v: int(np.sum(~np.isnan(v)))
    print(f'\n{name} ({len(images)} images, layer {conv_layer}, class index {class_index})')
    print(f'  Eager, {size} x {size}:        {s(t_e)}')
    print(f'  Compiled, {size} x {size}:     {s(t_c)} | speed-up {np.mean(t_e) / np.mean(t_c):.2f}x | '
          f'correlation {np.nanmean(c_c):.3f}, IoU {np.nanmean(i_c):.3f} ({n_ok(c_c)} images)')
    print(f'  Compiled, {reduced} x {reduced}:     {s(t_s)} | speed-up {np.mean(t_e) / np.mean(t_s):.2f}x | '
          f'correlation {np.nanmean(c_s):.3f}, IoU {np.nanmean(i_s):.3f} ({n_ok(c_s)} images)')
    print(f'  Images with a constant (all-zero) map: eager {flat_e}, reduced {flat_s}')
    del model, small, grad_full, grad_small
    tf.keras.backend.clear_session()

Clock adjustment factor: 1.47
NIH EfficientNetB4: the saved network does not accept 128 x 128 directly (ValueError)
NIH EfficientNetB4: largest output difference, rebuilt network against saved network at 224 x 224: 2.96e-08

NIH EfficientNetB4 (50 images, layer top_activation, class index 0)
  Eager, 224 x 224:        4.087 +/- 0.643 s
  Compiled, 224 x 224:     0.332 +/- 0.101 s | speed-up 12.29x | correlation 1.000, IoU 1.000 (50 images)
  Compiled, 128 x 128:     0.140 +/- 0.042 s | speed-up 29.24x | correlation 0.306, IoU 0.379 (50 images)
  Images with a constant (all-zero) map: eager 0, reduced 0
Mendeley Xception: the saved network does not accept 128 x 128 directly (ValueError)
Mendeley Xception: largest output difference, rebuilt network against saved network at 299 x 299: 0.00e+00

Mendeley Xception (50 images, layer block14_sepconv2_act, class index 1)
  Eager, 299 x 299:        3.282 +/- 0.682 s
  Compiled, 299 x 299:     0.675 +/- 0.216 s | speed-up 4.86x | correlation 1.0

### **Training-protocol audit of the training notebooks**

In [ ]:
# =============================================================================
# Training-protocol audit of the training notebooks
# =============================================================================
import glob
import json
import os
import re

import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = '/content/drive/MyDrive'
OUT_CSV = '/content/training_protocol_audit.csv'

FLAGS = {
    'EarlyStopping': r'EarlyStopping',
    'restore_best_weights': r'restore_best_weights\s*=\s*True',
    'ReduceLROnPlateau': r'ReduceLROnPlateau',
    'ModelCheckpoint': r'ModelCheckpoint',
    'rescale 1/255 (generator)': r'rescale\s*=\s*1\.?0?\s*/\s*255',
    'x/255 (dataset map)': r'\bx\s*/\s*255',
    'preprocess_input': r'preprocess_input',
    'horizontal_flip': r'horizontal_flip\s*=\s*True',
    'class_weight': r'class_weight',
    'focal loss': r'[Ff]ocal',
}
VALUES = {
    'rotation_range': r'rotation_range\s*=\s*([0-9.]+)',
    'width_shift_range': r'width_shift_range\s*=\s*([0-9.]+)',
    'zoom_range': r'zoom_range\s*=\s*([0-9.]+)',
    'shear_range': r'shear_range\s*=\s*([0-9.]+)',
    'validation_split': r'validation_split\s*=\s*([0-9.]+)',
    'epochs': r'(?:EPOCHS\w*|epochs)\s*=\s*(\d+)',
    'batch_size': r'(?:BATCH_SIZE|batch_size)\s*=\s*(\d+)',
}


def code_of(path):
    with open(path, encoding='utf-8') as f:
        nb = json.load(f)
    return '\n'.join(''.join(c.get('source', [])) for c in nb.get('cells', []) if c.get('cell_type') == 'code')


def dataset_of(code):
    found = []
    if re.search(r'NIH_processed|NIH_ChestXray|NIH_Models', code, re.I):
        found.append('NIH')
    if re.search(r'chest_xray', code, re.I):
        found.append('Mendeley')
    return '+'.join(found)


rows = []
for path in sorted(glob.glob(os.path.join(ROOT, '**', '*.ipynb'), recursive=True)):
    if '.ipynb_checkpoints' in path:
        continue
    try:
        code = code_of(path)
    except Exception as err:
        print(f'not read: {path} ({type(err).__name__})')
        continue
    if '.fit(' not in code:
        continue
    row = {'notebook': os.path.relpath(path, ROOT), 'dataset': dataset_of(code)}
    for name, pattern in FLAGS.items():
        row[name] = 'Y' if re.search(pattern, code) else ''
    for name, pattern in VALUES.items():
        row[name] = ','.join(sorted(set(re.findall(pattern, code)), key=float))
    rows.append(row)

table = pd.DataFrame(rows)
table.to_csv(OUT_CSV, index=False)
pd.set_option('display.width', 250, 'display.max_columns', None, 'display.max_colwidth', 70)
print(f'{len(table)} notebooks with a .fit( call under {ROOT}\n')
print(table.to_string(index=False))
print(f'\nSaved: {OUT_CSV}')

Mounted at /content/drive
41 notebooks with a .fit( call under /content/drive/MyDrive

                                                                      notebook      dataset EarlyStopping restore_best_weights ReduceLROnPlateau ModelCheckpoint rescale 1/255 (generator) x/255 (dataset map) preprocess_input horizontal_flip class_weight focal loss rotation_range width_shift_range zoom_range shear_range validation_split epochs batch_size
Colab Notebooks/10-Multi-Label Pneumonia Classification - EfficientNetB4.ipynb          NIH             Y                                      Y               Y                                                              Y                            Y          Y                                                                              30         32
 Colab Notebooks/2-Multi-Label Pneumonia Classification - EfficientNetB4.ipynb          NIH             Y                                      Y               Y                                           